## DuckDB + SLayer, the agent way (MCP)

Same demo as the [Python notebook](duckdb_python_nb.ipynb), but the way an **AI agent** experiences it: SLayer runs as a subprocess speaking the Model Context Protocol over stdio, and we drive it purely through its tools — the exact calls Claude would make.

In real use you register SLayer once and your agent spawns it on demand. With [uv](https://docs.astral.sh/uv/):

```bash
claude mcp add slayer -- uvx --from motley-slayer slayer mcp --storage /path/to/store
```

or as a JSON client config:

```json
{
  "mcpServers": {
    "slayer": {
      "command": "slayer",
      "args": ["mcp", "--storage", "/path/to/store"]
    }
  }
}
```

Here we spawn it in-notebook and connect as a stdio client.

## 1. Point DuckDB at the file online

Same remote view as the Python notebook, in this notebook's own cache dir. The MCP server will open this storage folder and read the view over `httpfs`.

In [1]:
import shutil
from pathlib import Path

import duckdb

CACHE = Path(".cache/mcp")
shutil.rmtree(CACHE, ignore_errors=True)
CACHE.mkdir(parents=True)

DB_PATH = (CACHE / "weather.duckdb").resolve()
MODELS_DIR = (CACHE / "models").resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CSV_URL = "https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv"

con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE VIEW weather AS SELECT * FROM '{CSV_URL}'")
assert con.sql("SELECT count(*) FROM weather").fetchone()[0] == 1461
con.close()
print("view ready:", DB_PATH.relative_to(Path.cwd()))

view ready: .cache/mcp/weather.duckdb


## 2. Start `slayer mcp` and connect over stdio

We launch `slayer mcp --storage <dir>` as a subprocess and open an MCP session against it. A stdio session has to be opened and closed inside one async task, but each notebook cell runs in its own task — so the little `MCPServer` wrapper runs the session on a background event loop and exposes a plain synchronous `server.call(tool, **args)` the rest of the notebook uses. Re-running this cell stops any server a previous run left behind.

In [2]:
import asyncio
import concurrent.futures
import os
import threading
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.types import CallToolResult, TextContent


class MCPServer:
    """A `slayer mcp` subprocess, driven synchronously from notebook cells.

    A stdio MCP session must be opened and closed in one asyncio task, but each
    notebook cell runs in its own task. So the session lives on a background
    event loop; cells submit a tool call and block for its result.
    """

    def __init__(self, storage_dir):
        self._loop = asyncio.new_event_loop()
        self._inbox = None
        self._ready = threading.Event()
        self._error = None
        self._stopped = False
        self._thread = threading.Thread(
            target=self._loop.run_until_complete,
            args=(self._serve(str(storage_dir)),),
            daemon=True,
        )
        self._thread.start()
        self._ready.wait()
        if self._error is not None:
            raise self._error

    async def _serve(self, storage_dir):
        try:
            self._inbox = asyncio.Queue()
            params = StdioServerParameters(
                command="slayer", args=["mcp", "--storage", storage_dir]
            )
            with open(os.devnull, "w") as devnull:
                async with AsyncExitStack() as stack:
                    read, write = await stack.enter_async_context(
                        stdio_client(params, errlog=devnull)
                    )
                    session = await stack.enter_async_context(ClientSession(read, write))
                    await session.initialize()
                    self._ready.set()
                    while True:
                        make_coro, future = await self._inbox.get()
                        if make_coro is None:
                            future.set_result(None)
                            break
                        try:
                            future.set_result(await make_coro(session))
                        except Exception as exc:
                            future.set_exception(exc)
        except Exception as exc:
            self._error = exc
        finally:
            self._ready.set()

    def _submit(self, make_coro):
        if self._stopped or not self._thread.is_alive():
            raise RuntimeError("slayer mcp session is not running")
        future = concurrent.futures.Future()
        self._loop.call_soon_threadsafe(self._inbox.put_nowait, (make_coro, future))
        return future.result(timeout=120)  # fast-fail if the session dies mid-call

    def list_tools(self):
        return self._submit(lambda s: s.list_tools()).tools

    def call(self, tool, /, **arguments):
        """Invoke a SLayer MCP tool and return its text response."""
        result: CallToolResult = self._submit(
            lambda s: s.call_tool(name=tool, arguments=arguments)
        )
        text = "\n".join(b.text for b in result.content if isinstance(b, TextContent))
        if result.isError:
            raise RuntimeError(f"MCP tool {tool!r} failed:\n{text}")
        return text

    def stop(self):
        if self._stopped or not self._thread.is_alive():
            self._stopped = True
            return
        self._stopped = True
        future = concurrent.futures.Future()
        self._loop.call_soon_threadsafe(self._inbox.put_nowait, (None, future))
        future.result(timeout=30)
        self._thread.join(timeout=5)


# Rerun-safe: stop a server left running by a previous run of this cell.
if "server" in globals():
    try:
        server.stop()
    except Exception:
        pass

server = MCPServer(MODELS_DIR)
print(f"slayer mcp exposes {len(server.list_tools())} tools")

slayer mcp exposes 21 tools


## 3. Register the datasource and ingest — two tool calls

`create_datasource` with `auto_ingest=False` registers the DuckDB connection without ingesting, so the separate `ingest_datasource_models` call shows the schema being read live.

In [3]:
print(server.call(
    "create_datasource",
    name="weather_db",
    type="duckdb",
    database=str(DB_PATH),
    auto_ingest=False,
))
print(server.call("ingest_datasource_models", datasource_name="weather_db"))

Datasource 'weather_db' created. Connection successful.


Created 1 new model(s):
- weather (6 columns, 0 joins)


## 4. Look at the model

`models_summary` is how an agent orients itself before querying.

In [4]:
print(server.call("models_summary", datasource_name="weather_db"))

# Datasource: `weather_db` — 1 model(s)

## `weather`
Columns: 6
Measures: 
Joins to: _(none)_


## 5. A warm-up query

The `query` tool takes typed measures and dimensions and returns a rendered table.

In [5]:
print(server.call(
    "query",
    source_model="weather",
    dimensions=["weather"],
    measures=[
        {"formula": "temp_max:avg", "name": "avg_high"},
        {"formula": "*:count", "name": "days"},
    ],
    order=[{"column": "days", "direction": "desc"}],
))

| weather.weather | weather.avg_high | weather.days |
| --- | --- | --- |
| rain | 13.5 | 641 |
| sun | 19.9 | 640 |
| fog | 16.8 | 101 |
| drizzle | 15.9 | 53 |
| snow | 5.57 | 26 |

Measure attributes:
  weather.avg_high: format=(type=float)
  weather.days: format=(type=integer)


## 6. The hero query

The two-stage query — an aggregate used as a `CASE WHEN` dimension (rainy vs dry month), plus `time_shift` for change versus the same month last year — goes through the `query_nested` tool, which runs a list of stages as one DAG. The first year's year-over-year values are null: nothing precedes 2012.

In [6]:
hero = [
    {
        "name": "monthly",
        "source_model": "weather",
        "time_dimensions": [{"dimension": "date", "granularity": "month"}],
        "measures": [
            {"formula": "precipitation:sum", "name": "rain"},
            {
                "formula": "precipitation:sum - time_shift(precipitation:sum, -1, 'year')",
                "name": "rain_yoy",
            },
        ],
    },
    {
        "source_model": "monthly",
        "dimensions": [
            {
                "expression": "CASE WHEN rain > 100 THEN 'rainy' ELSE 'dry' END",
                "name": "month_type",
            },
            "date",
        ],
        "measures": [
            {"formula": "rain:sum", "name": "total_rain"},
            {"formula": "rain_yoy:sum", "name": "total_rain_yoy"},
        ],
        "order": [{"column": "date", "direction": "asc"}],
    },
]

hero_md = server.call("query_nested", queries=hero)
month_rows = [
    r for r in hero_md.splitlines()
    if r.startswith("| rainy ") or r.startswith("| dry ")
]
assert len(month_rows) == 48, f"expected 48 month rows, got {len(month_rows)}"
print(hero_md)

| monthly.month_type | monthly.date | monthly.total_rain | monthly.total_rain_yoy |
| --- | --- | --- | --- |
| rainy | 2012-01-01 00:00:00 | 173.29999999999998 |  |
| dry | 2012-02-01 00:00:00 | 92.3 |  |
| rainy | 2012-03-01 00:00:00 | 183.0 |  |
| dry | 2012-04-01 00:00:00 | 68.09999999999998 |  |
| dry | 2012-05-01 00:00:00 | 52.199999999999996 |  |
| dry | 2012-06-01 00:00:00 | 75.1 |  |
| dry | 2012-07-01 00:00:00 | 26.3 |  |
| dry | 2012-08-01 00:00:00 | 0.0 |  |
| dry | 2012-09-01 00:00:00 | 0.8999999999999999 |  |
| rainy | 2012-10-01 00:00:00 | 170.29999999999998 |  |
| rainy | 2012-11-01 00:00:00 | 210.5 |  |
| rainy | 2012-12-01 00:00:00 | 174.0 |  |
| rainy | 2013-01-01 00:00:00 | 105.69999999999997 | -67.60000000000001 |
| dry | 2013-02-01 00:00:00 | 40.300000000000004 | -51.99999999999999 |
| dry | 2013-03-01 00:00:00 | 69.7 | -113.3 |
| rainy | 2013-04-01 00:00:00 | 149.60000000000002 | 81.50000000000004 |
| dry | 2013-05-01 00:00:00 | 60.49999999999999 | 8.299999999999

## 7. The SQL that ran

`dry_run=True` returns the single generated SQL statement without executing it — the aggregate CTE, the year-shifted self-join, the banding and regroup on top.

In [7]:
hero_sql = server.call("query_nested", queries=hero, dry_run=True)
assert "SELECT" in hero_sql, "expected SQL in the dry-run response"
print(hero_sql)

SQL:
WITH monthly AS (
  SELECT
    _stage_inner."weather.date" AS "date",
    _stage_inner."weather.rain" AS "rain",
    _stage_inner."weather.rain_yoy" AS "rain_yoy"
  FROM (
    SELECT
      "weather.date",
      "weather.rain",
      "weather.rain_yoy"
    FROM (
      WITH base AS (
        SELECT
          DATE_TRUNC('MONTH', weather.date) AS "weather.date",
          CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.rain"
        FROM weather AS weather
        GROUP BY
          DATE_TRUNC('MONTH', weather.date)
      ), shifted__time_shift_inner AS (
        SELECT
          DATE_TRUNC('MONTH', weather.date) + INTERVAL '1' YEAR AS "weather.date",
          CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.rain"
        FROM weather AS weather
        GROUP BY
          DATE_TRUNC('MONTH', weather.date) + INTERVAL '1' YEAR
      ), sjoin__time_shift_inner AS (
        SELECT
          base."weather.date",
          base."weather.rain",
          shifted__time_shift_in

## 8. Stop the server

`server.stop()` shuts down the session and the `slayer mcp` subprocess, releasing the DuckDB file.

In [8]:
server.stop()
print("slayer mcp stopped")

slayer mcp stopped


---

Same result as the [Python notebook](duckdb_python_nb.ipynb) — a semantic layer over a file on the internet — reached entirely through the tools an agent calls. See [Introspecting a datasource via MCP](../08_mcp_introspect/mcp_introspect_nb.ipynb) for a deeper tour of the MCP tools.